In [ ]:
%%configure -f
{ "conf":{
    "spark.pyspark.python": "python3", 
    "spark.pyspark.virtualenv.enabled": "true", 
    "spark.pyspark.virtualenv.type": "native", 
    "spark.pyspark.virtualenv.bin.path": "/xxxx/xxx/xxxxxx",
    "spark.executor.instances":"2",
    "spark.dynamicAllocation.enabled":"true",
    "spark.submit.pyFiles":"xxxxxxx"
}
}

In [ ]:
import os
import json
import re
from difflib import get_close_matches
import datetime
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import udf,col,lit,when,coalesce,expr,count,countDistinct,to_date
import pandas as pd

## master dict

In [ ]:
# dictionary

s3 = boto3.resource('s3')
bucket_dict = 'xxxx-bucket'

# gender
dict_gender = 'gender'
input_gender = 'Data Engineer Intern at BTS Group (Jul - Nov 2023)/Data Quality Improvement Project/master_dict/{0}_pattern.json'.format(dict_gender)
content_object_gender = s3.Object(bucket_dict, input_gender)
file_content_gender = content_object_gender.get()['Body'].read().decode('utf-8')
json_content_gender = json.loads(file_content_gender)

# marital status
dict_marital_status = 'marital_status'
input_marital_status = 'Data Engineer Intern at BTS Group (Jul - Nov 2023)/Data Quality Improvement Project/master_dict/{0}_pattern.json'.format(dict_marital_status)
content_object_marital_status = s3.Object(bucket_dict, input_marital_status)
file_content_marital_status = content_object_marital_status.get()['Body'].read().decode('utf-8')
json_content_marital_status = json.loads(file_content_marital_status)

# education
dict_education = 'education'
input_education = 'Data Engineer Intern at BTS Group (Jul - Nov 2023)/Data Quality Improvement Project/master_dict/{0}_pattern.json'.format(dict_education)
content_object_education = s3.Object(bucket_dict, input_education)
file_content_education = content_object_education.get()['Body'].read().decode('utf-8')
json_content_education = json.loads(file_content_education)

# occupation
dict_occupation = 'occupation'
input_occupation = 'Data Engineer Intern at BTS Group (Jul - Nov 2023)/Data Quality Improvement Project/master_dict/{0}_pattern.json'.format(dict_occupation)
content_object_occupation = s3.Object(bucket_dict, input_occupation)
file_content_occupation = content_object_occupation.get()['Body'].read().decode('utf-8')
json_content_occupation = json.loads(file_content_occupation)

# nationality
dict_nationality = 'nationality'
input_nationality = 'Data Engineer Intern at BTS Group (Jul - Nov 2023)/Data Quality Improvement Project/master_dict/{0}_pattern.json'.format(dict_nationality)
content_object_nationality = s3.Object(bucket_dict, input_nationality)
file_content_nationality = content_object_nationality.get()['Body'].read().decode('utf-8')
json_content_nationality = json.loads(file_content_nationality)

# birth date
dict_birth_date = 'date_of_birth'
input_birth_date = 'Data Engineer Intern at BTS Group (Jul - Nov 2023)/Data Quality Improvement Project/master_dict/{0}_pattern.json'.format(dict_birth_date)
content_object_birth_date = s3.Object(bucket_dict, input_birth_date)
file_content_birth_date = content_object_birth_date.get()['Body'].read().decode('utf-8')
json_content_birth_date = json.loads(file_content_birth_date)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## dataset

In [ ]:
# data
bucket_name = 'data-quality-improvement'
input_key = 'dataset/'

s3_input_path = 's3://{0}/{1}'.format(bucket_name,input_key)
df = spark.read.options(header=True).parquet(s3_input_path).persist()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
df = df.select(col("customer_sk").alias("customer_id"),
               col("birth_date").alias("birth_date"),
               col("gender").alias("gender"),
               col("marital_status").alias("marital_status"),
               col("occupation").alias("occupation"),
               col("birth_country").alias("nationality"), 
               col("income").alias("income"),
               col("education").alias("education"))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [8]:
df.count()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2824766

### gender

In [9]:
data = 'gender'

# make the data to lower case
# remove .
# Consecutively characters convert to one-character
def standard_transform(text_input):
    if text_input == None :
        return '-'
    
    text_input = text_input.lower().replace('.' , '')
    # [a-zก-๛0-9]
    text_input = re.sub(r'([\u0E00-\u0E7F0-9])\1+' , r'\1' , text_input)
    return text_input


udf_standard_transform = udf(standard_transform,StringType())

df = df.withColumn('gender_transform',udf_standard_transform(col(data)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
data_transform = 'gender_transform'

# choose only english letters
def take_eng(text_input) :
    try :
        result = re.search(r'[a-zA-Z\-]+' , text_input).group()    
    except AttributeError :
        return None
    return result


udf_take_eng = udf(take_eng,StringType())

df = df.withColumn('df_eng',udf_take_eng(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
# choose only thai letters
def take_thai(text_input) :
    try :
        # [ก-๛0-9]
        result = re.search(r'[\u0E00-\u0E7F0-9]+' , text_input).group()
    except AttributeError :
        return None
    return result


udf_take_thai = udf(take_thai,StringType())

df = df.withColumn('df_thai',udf_take_thai(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
# create data frame for case = eng and case = thai

gender_eng = df.filter(df['df_eng'].isNotNull())
gender_thai = df.filter(df['df_thai'].isNotNull())

df = df.drop('df_eng', 'df_thai', 'gender_transform')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
data_en = 'df_eng'
data_th = 'df_thai'

# create standard format
def std_gend(text_input, lang):
    try :
        if not isinstance(text_input, str):
            return None

        for pattern,value in json_content_gender[lang].items():
            if re.search(pattern , text_input):
                return value
            else:
                continue
        return None
    
    except :
        return text_input



udf_std_gend = udf(std_gend,StringType())

gender_eng = gender_eng.withColumn('standard_gender', udf_std_gend(col(data_en), lit('eng')))

gender_thai = gender_thai.withColumn('standard_gender', udf_std_gend(col(data_th), lit('tha')))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
# combined dataframe
selected_cols = ['customer_id', 'gender', 'gender_transform' , 'standard_gender']

gender = (gender_eng.select(*selected_cols).union(gender_thai.select(*selected_cols))).dropDuplicates(["customer_id"])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
data = 'gender'
data_standard = 'standard_gender'

# standard output / status
def result(text_input, standard):
    if text_input is None and standard is None:
        return text_input, 3
    elif isinstance(text_input, str) and isinstance(standard, str):
        if text_input == standard:
            return text_input, 1
        else:
            return standard, 2
    elif isinstance(text_input, str) and standard == None :
        return text_input, 0
    

    
schema = StructType([
    StructField("std_output", StringType(), True),
    StructField("status", IntegerType(), True)
])

udf_result = udf(result, schema)

# Apply the UDF to your DataFrame
gender = gender.withColumn("segment_data", udf_result(col(data), col(data_standard)))

# Split the struct column into separate columns
gender = gender.withColumn("std_output", gender["segment_data.std_output"])
gender = gender.withColumn("status", gender["segment_data.status"])

# Drop the intermediate struct column
gender = gender.drop("segment_data")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
gender.show()

## marital status

In [17]:
data = 'marital_status'

# make the data to lower case
# remove .
# Consecutively characters convert to one-character
def standard_transform(text_input):
    if text_input == None :
        return '-'
    
    text_input = text_input.lower().replace('.' , '')
    # [a-zก-๛0-9]
    text_input = re.sub(r'([a-z\u0E00-\u0E7F0-9])\1+' , r'\1' , text_input)
    return text_input


udf_standard_transform = udf(standard_transform,StringType())

df = df.withColumn('marital_transform',udf_standard_transform(col(data)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [18]:
data_transform = 'marital_transform'

# choose only english letters
def take_eng(text_input) :
    try :
        if re.search(r'[a-zA-Z\-]+' , text_input) :
            return text_input
    except AttributeError :
        return None


udf_take_eng = udf(take_eng,StringType())

df = df.withColumn('df_eng',udf_take_eng(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
# choose only thai letters
def take_thai(text_input) :
    try :
        # [ก-๛]
        if re.search(r'[\u0E00-\u0E7F]+' , text_input) :
            return text_input
    except AttributeError :
        return None


udf_take_thai = udf(take_thai,StringType())

df = df.withColumn('df_thai',udf_take_thai(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [20]:
# create data frame for case = eng and case = thai

marital_eng = df.filter(df['df_eng'].isNotNull())
marital_thai = df.filter(df['df_thai'].isNotNull())

df = df.drop('df_eng', 'df_thai', 'marital_transform')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [21]:
data_en = 'df_eng'
data_th = 'df_thai'

# create standard format
def std_marital(text_input, lang):
    if not isinstance(text_input, str):
        return None

    for pattern,value in json_content_marital_status[lang].items():
        if re.search(pattern , text_input):
            return value
        else:
            continue
    return None



udf_std_marital = udf(std_marital,StringType())

marital_eng = marital_eng.withColumn('standard_marital_status', udf_std_marital(col(data_en), lit('eng')))

marital_thai = marital_thai.withColumn('standard_marital_status', udf_std_marital(col(data_th), lit('tha')))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [22]:
# combined dataframe
selected_cols = ['customer_id', 'marital_status','marital_transform' ,'standard_marital_status']

marital_status = (marital_eng.select(*selected_cols).union(marital_thai.select(*selected_cols))).dropDuplicates(["customer_id"])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
data = 'marital_status'
data_standard = 'standard_marital_status'

# standard output / status
def result(text_input, standard):
    if text_input is None and standard is None:
        return text_input, 3
    elif isinstance(text_input, str) and isinstance(standard, str):
        if text_input == standard:
            return text_input, 1
        else:
            return standard, 2
    elif isinstance(text_input, str) and standard == None :
        return text_input, 0
    

    
schema = StructType([
    StructField("std_output", StringType(), True),
    StructField("status", IntegerType(), True)
])

udf_result = udf(result, schema)

# Apply the UDF to your DataFrame
marital_status = marital_status.withColumn("segment_data", udf_result(col(data), col(data_standard)))

# Split the struct column into separate columns
marital_status = marital_status.withColumn("std_output", marital_status["segment_data.std_output"])
marital_status = marital_status.withColumn("status", marital_status["segment_data.status"])

# Drop the intermediate struct column
marital_status = marital_status.drop("segment_data")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
marital_status.show()

## education

In [25]:
data = 'education'

# make the data to lower case
# remove . and white space
def standard_transform(text_input):
    if text_input == None :
        return '-'
    
    text_input = text_input.lower().replace('.' , '').replace(' ' , '')
    return text_input


udf_standard_transform = udf(standard_transform,StringType())

df = df.withColumn('education_transform',udf_standard_transform(col(data)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [26]:
data_transform = 'education_transform'

# choose only english letters
def take_eng(text_input) :
    try :
        result = re.search(r'[a-zA-Z0-9\-_ ]+' , text_input).group()    
    except AttributeError :
        return None
    return result


udf_take_eng = udf(take_eng,StringType())

df = df.withColumn('df_eng',udf_take_eng(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [27]:
# choose only thai letters
def take_thai(text_input) :
    try :
        # [ก-๛0-9]
        result = re.search(r'[\u0E00-\u0E7F0-9]+' , text_input).group()
    except AttributeError :
        return None
    return result


udf_take_thai = udf(take_thai,StringType())

df = df.withColumn('df_thai',udf_take_thai(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [28]:
# create data frame for case = eng and case = thai

education_eng = df.filter(df['df_eng'].isNotNull())
education_thai = df.filter(df['df_thai'].isNotNull())

df = df.drop('df_eng', 'df_thai', 'education_transform')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [29]:
data_en = 'df_eng'
data_th = 'df_thai'

# create standard format
def std_edu(text_input, lang):
    if not isinstance(text_input, str):
        return None

    for pattern,value in json_content_education[lang].items():
        if re.search(pattern , text_input):
            return value
        else:
            continue
    return None



udf_std_edu = udf(std_edu,StringType())

education_eng = education_eng.withColumn('standard_education', udf_std_edu(col(data_en), lit('eng')))

education_thai = education_thai.withColumn('standard_education', udf_std_edu(col(data_th), lit('tha')))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [30]:
# combined dataframe
selected_cols = ['customer_id','education', 'education_transform','standard_education']

education = (education_eng.select(*selected_cols).union(education_thai.select(*selected_cols))).dropDuplicates(["customer_id"])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [31]:
data = 'education'
data_standard = 'standard_education'

# standard output / status
def result(text_input, standard):
    if text_input is None and standard is None:
        return text_input, 3
    elif isinstance(text_input, str) and isinstance(standard, str):
        if text_input == standard:
            return text_input, 1
        else:
            return standard, 2
    elif isinstance(text_input, str) and standard == None :
        return text_input, 0
    

    
schema = StructType([
    StructField("std_output", StringType(), True),
    StructField("status", IntegerType(), True)
])

udf_result = udf(result, schema)

# Apply the UDF to your DataFrame
education = education.withColumn("segment_data", udf_result(col(data), col(data_standard)))

# Split the struct column into separate columns
education = education.withColumn("std_output", education["segment_data.std_output"])
education = education.withColumn("status", education["segment_data.status"])

# Drop the intermediate struct column
education = education.drop("segment_data")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
education.show()

## occupation

In [33]:
data = 'occupation'

# make the data to lower case
# remove . and ,
# replace - to white space
def standard_transform(text_input):
    if text_input == None :
        return '-'
    
    text_input = text_input.lower().replace('.' , '').replace('-' , ' ').replace(',' , '')
    return text_input


udf_standard_transform = udf(standard_transform,StringType())

df = df.withColumn('occupation_transform',udf_standard_transform(col(data)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [34]:
data_transform = 'occupation_transform'

# define type of occupations are 'business owner' , 'employee' , 'officer' , 'other'
def typ(text_input) :
    # ^(b|ow.*|ow.*[0-9]\s?[a-z]|เ.*จด|ธ.*จด|เ.*[0-9]|ธ.*[0-9])+
    if re.search(r'^(b|ow.*|ow.*[0-9]\s?[a-z]|\u0E40.*\u0E08\u0E14|\u0E18.*\u0E08\u0E14|\u0E40.*[0-9]|\u0E18.*[0-9])+' , text_input) :
        return 'business_owner'
    
    # ^(em|ล)
    elif re.search(r'^(em|\u0E25)' , text_input) :
        return 'employee'
    
    # ^(of|พ)
    elif re.search(r'^(of|\u0E1E)' , text_input) :
        return 'officer'
    
    elif text_input == '-' :
        return None
    
    else :
        return 'other'

    
    
udf_typ = udf(typ,StringType())

df = df.withColumn('type',udf_typ(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [35]:
# choose only english letters
def take_eng(text_input) :
    try :
        if re.search(r'[a-zA-Z\-]+' , text_input) :
            return text_input
    except AttributeError :
        return None
    return None


udf_take_eng = udf(take_eng,StringType())

df = df.withColumn('df_eng',udf_take_eng(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [36]:
# choose only thai letters
def take_thai(text_input) :
    try :
        # [ก-๛]
        if re.search(r'[\u0E00-\u0E7F]+' , text_input) :
                return text_input
    except AttributeError :
        return None


udf_take_thai = udf(take_thai,StringType())

df = df.withColumn('df_thai',udf_take_thai(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [37]:
# create data frame for case = eng and case = thai

eng_business_owner = df.filter((df['df_eng'].isNotNull()) & (df['type'] == 'business_owner'))
eng_employee = df.filter((df['df_eng'].isNotNull()) & (df['type'] == 'employee'))
eng_officer = df.filter((df['df_eng'].isNotNull()) & (df['type'] == 'officer'))
eng_other = df.filter((df['df_eng'].isNotNull()) & (df['type'] == 'other'))
eng_unknown = df.filter((df['df_eng'].isNotNull()) & (df['type'].isNull()))

tha_business_owner = df.filter((df['df_thai'].isNotNull()) & (df['type'] == 'business_owner'))
tha_employee = df.filter((df['df_thai'].isNotNull()) & (df['type'] == 'employee'))
tha_officer = df.filter((df['df_thai'].isNotNull()) & (df['type'] == 'officer'))
tha_other = df.filter((df['df_thai'].isNotNull()) & (df['type'].isNull()))


df = df.drop('df_eng', 'df_thai', 'occupation_transform', 'type')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [38]:
data_en = 'df_eng'
data_th = 'df_thai'

# create standard format
def std_occupation(text_input, lang, typ):
    if text_input == '-' :
        return None
    
    for pattern, value in json_content_occupation[lang][typ].items():
        if re.search(pattern, text_input):
            return value
    return None



udf_std_occupation = udf(std_occupation,StringType())

# Add a new column for standard_occupation for 'eng' and 'tha'
eng_business_owner = eng_business_owner.withColumn('standard_occupation', udf_std_occupation(col(data_en), lit('eng'), lit('business owner')))
eng_employee = eng_employee.withColumn('standard_occupation', udf_std_occupation(col(data_en), lit('eng'), lit('employee')))
eng_officer = eng_officer.withColumn('standard_occupation', udf_std_occupation(col(data_en), lit('eng'), lit('officer')))
eng_other = eng_other.withColumn('standard_occupation', udf_std_occupation(col(data_en), lit('eng'), lit('other')))
eng_unknown = eng_unknown.withColumn('standard_occupation', udf_std_occupation(col(data_en), lit('eng'), lit(None)))


tha_business_owner = tha_business_owner.withColumn('standard_occupation', udf_std_occupation(col(data_th), lit('tha'), lit('business owner')))
tha_employee = tha_employee.withColumn('standard_occupation', udf_std_occupation(col(data_th), lit('tha'), lit('employee')))
tha_officer = tha_officer.withColumn('standard_occupation', udf_std_occupation(col(data_th), lit('tha'), lit('officer')))
tha_other = tha_other.withColumn('standard_occupation', udf_std_occupation(col(data_th), lit('tha'), lit('other')))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [39]:
# # combined dataframe
selected_cols = ['customer_id','occupation', 'occupation_transform', 'standard_occupation']

occupation = (eng_business_owner.select(*selected_cols)
              .union(eng_employee.select(*selected_cols))
              .union(eng_officer.select(*selected_cols))
              .union(eng_other.select(*selected_cols))
              .union(eng_unknown.select(*selected_cols))
              .union(tha_business_owner.select(*selected_cols))
              .union(tha_employee.select(*selected_cols))
              .union(tha_officer.select(*selected_cols))
              .union(tha_other.select(*selected_cols))).dropDuplicates(["customer_id"])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [40]:
data = 'occupation'
data_standard = 'standard_occupation'

# standard output / status
def result(text_input, standard):
    if text_input is None and standard is None:
        return text_input, 3
    elif isinstance(text_input, str) and isinstance(standard, str):
        if text_input == standard:
            return text_input, 1
        else:
            return standard, 2
    elif isinstance(text_input, str) and standard == None :
        return text_input, 0
    

    
schema = StructType([
    StructField("std_output", StringType(), True),
    StructField("status", IntegerType(), True)
])

udf_result = udf(result, schema)

# Apply the UDF to your DataFrame
occupation = occupation.withColumn("segment_data", udf_result(col(data), col(data_standard)))

# Split the struct column into separate columns
occupation = occupation.withColumn("std_output", occupation["segment_data.std_output"])
occupation = occupation.withColumn("status", occupation["segment_data.status"])

# Drop the intermediate struct column
occupation = occupation.drop("segment_data")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
occupation.show()

## nationality

In [42]:
data_transform = 'nationality'

# choose only english letters
# remove . and _
# make the data to title case
def take_eng(text_input) :
    try :
        if text_input == None :
            return '-'
    
        if re.search(r'[a-zA-Z]{3,}' , text_input):
            text_input = text_input.replace('_' , '').replace('.' , '').title()
            return text_input
    except AttributeError :
        return None


    
    
udf_take_eng = udf(take_eng,StringType())

df = df.withColumn('df_eng',udf_take_eng(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [43]:
# choose only thai letters
# remove . and _ and white space
def take_thai(text_input) :
    try :
        if text_input == None :
            return '-'
        
        # [ก-๛]
        if re.search(r'[\u0E00-\u0E7F]+' , text_input) :
            text_input = text_input.replace('_' , '').replace(' ' , '').replace('.' , '')
            return text_input
    except AttributeError :
        return None

    
    
udf_take_thai = udf(take_thai,StringType())

df = df.withColumn('df_thai',udf_take_thai(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [44]:
# choose only country code
# remove . and _ and - and white space
# make the data to upper case
def take_country_code(text_input) :
    try :
        if text_input == None :
            return '-'
        
        if re.search(r'^[a-zA-Z]{2}$' , text_input):
            text_input = text_input.upper().replace('_' , '').replace(' ' , '').replace('.' , '').replace('-' , '')
            return text_input
    except AttributeError :
        return None
    
    
    
udf_take_country_code = udf(take_country_code,StringType())

df = df.withColumn('df_countrycode',udf_take_country_code(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [45]:
# create data frame for english letters, thai letters and country code

nationality_eng = df.filter(df['df_eng'].isNotNull())
nationality_thai = df.filter(df['df_thai'].isNotNull())
nationality_countrycode = df.filter(df['df_countrycode'].isNotNull())

df = df.drop('df_eng', 'df_thai', 'df_countrycode', 'nationality_transform')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [46]:
data_en = 'df_eng'

# create standard format
# english letter
# for data didn't be in dictionary find similar word of key and data
def std_nat_en(text_input, name_en, person, nationality):
    try:
        typ = [name_en, person, nationality]
        
        for case in typ:
            if case in json_content_nationality and text_input in json_content_nationality[case]:
                result = json_content_nationality[case][text_input]
                return result
            else:
                if case in json_content_nationality:
                    similar_keys = get_close_matches(text_input, json_content_nationality[case].keys(), n=1, cutoff=0.7)
                    if similar_keys:
                        value = json_content_nationality[case][similar_keys[0]]
                        return value
        
        return None
    except KeyError:
        return None



udf_std_nat_en = udf(std_nat_en,StringType())

# english letters
nationality_eng = nationality_eng.withColumn('standard_nationality', udf_std_nat_en(col(data_en), lit('country_nationality'), lit('person_nationality'), lit('nationality_nationality')))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [47]:
data_countrycode = 'df_countrycode'
data_th = 'df_thai'

# create standard format
# conutrycode and thai letter
# for data didn't be in dictionary find similar word of key and data
def std_nat(text_input, case):
    try:
        if text_input == None :
            return None
        
        if text_input in json_content_nationality[case]:
            result = json_content_nationality[case][text_input]
            return result
        else:
            similar_keys = get_close_matches(text_input, json_content_nationality[case].keys(), n=1, cutoff=0.7)
            if similar_keys:
                value = json_content_nationality[case][similar_keys[0]]
                return value
            else:
                return None
    except KeyError:
        return None



udf_std_nat = udf(std_nat,StringType())

# country code
nationality_countrycode = nationality_countrycode.withColumn('standard_nationality', udf_std_nat(col(data_countrycode), lit('contrycode_country')))
nationality_countrycode = nationality_countrycode.withColumn('standard_nationality', udf_std_nat(col('standard_nationality'), lit('country_nationality')))

# thai letters
nationality_thai = nationality_thai.withColumn('standard_nationality', udf_std_nat(col(data_th), lit('country_thai_eng')))
nationality_thai = nationality_thai.withColumn('standard_nationality', udf_std_nat(col('standard_nationality'), lit('country_nationality')))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [48]:
# combined dataframe
selected_cols = ['customer_id','nationality', 'nationality_transform','standard_nationality']

nationality = (nationality_eng.union(nationality_countrycode).union(nationality_thai)).dropDuplicates(["customer_id"])

nationality = nationality.withColumn('nationality_transform',coalesce(col('df_eng'), col('df_countrycode'), col('df_thai')))

nationality = nationality.select(*selected_cols)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [49]:
data = 'nationality'
data_standard = 'standard_nationality'

# standard output / status
def result(text_input, standard):
    if text_input is None and standard is None:
        return text_input, 3
    elif isinstance(text_input, str) and isinstance(standard, str):
        if text_input == standard:
            return text_input, 1
        else:
            return standard, 2
    elif isinstance(text_input, str) and standard == None:
        return text_input, 0
    

    
schema = StructType([
    StructField("std_output", StringType(), True),
    StructField("status", IntegerType(), True)
])

udf_result = udf(result, schema)

# Apply the UDF to your DataFrame
nationality = nationality.withColumn("segment_data", udf_result(col(data), col(data_standard)))

# Split the struct column into separate columns
nationality = nationality.withColumn("std_output", nationality["segment_data.std_output"])
nationality = nationality.withColumn("status", nationality["segment_data.status"])

# Drop the intermediate struct column
nationality = nationality.drop("segment_data")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
nationality.show()

## income

In [51]:
data = 'income'

# remove ,
# convert T | – - to -
# convert x k m to upper case
def standard_transform(text_input): 
    replace = {
        'to': ' - ',
        'T': ' - ',
        '|': ' - ',
        'X': 'K',
        'k' : 'K',
        'm$': 'M',
        '–': ' - ',
        '-': ' - ',
        ',': ''
    }
    
    for old, new in replace.items():
        text_input = text_input.replace(old, new)
    
    # convert K to 1000 , M to 1000000
    try :
        text_input = re.sub(r'[ก-๛]' , '' , text_input)
        if re.search(r' - ' , text_input) :
            split = text_input.split(' - ')

            min_value = split[0].strip()
            max_value = split[1].strip()

            if min_value.endswith('K'):
                result_min = int(float(min_value[:-1]) * 1000)
            elif min_value.endswith('M'):
                result_min = int(float(min_value[:-1]) * 1000000)
            else :
                result_min = re.sub(r'\.[0-9]+$', '', min_value).replace('.' , '')

            if max_value.endswith('K'):
                result_max = int(float(max_value[:-1]) * 1000)
            elif max_value.endswith('M'):
                result_max = int(float(max_value[:-1]) * 1000000)                
            else:
                result_max = re.sub(r'\.[0-9]+$', '', max_value).replace('.' , '')

            result = f'{result_min} - {result_max}'
            return result
        

        def convert_char(text_input):
            if '.' in text_input.group():
                return str(int(float(text_input.group())))
            elif 'K' in text_input.group():
                return str(int(float(text_input.group()[:-1]) * 1000))
            elif 'M' in text_input.group():
                return str(int(float(text_input.group()[:-1]) * 1000000))
            else:
                return text_input.group()
            
        result = re.sub(r'(\d+(\.\d+)?K?M?)', convert_char, text_input).lower().replace('.' , '')
        return result

    except :
        return None


udf_standard_transform = udf(standard_transform,StringType())

df = df.withColumn('income_transform',udf_standard_transform(col(data)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [52]:
data_transform = 'income_transform'

# switch position of some data
def switch_range(text_input) :
    try :
        split = text_input.split('-')
        
        min_value = int(split[0])
        max_value = int(split[1])
        
        if min_value > max_value :
            result = f"{max_value} - {min_value}"
            return result
        else :
            return text_input
    except :
        return text_input
    
    
udf_switch_range = udf(switch_range,StringType())

df = df.withColumn('income_transform',udf_switch_range(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [53]:
# choose only range
def take_range(text_input):
    if text_input is None:
        return None

    try:
        if re.search(r'([0-9a-z]{1,10}\s?-\s?[0-9a-z]{1,10})', text_input):
            split = text_input.split(' - ')
            
            min_value = str(split[0])
            max_value = str(split[1])

            if min_value != max_value :
                return f"{min_value} - {max_value}"
            
    except AttributeError:
        return None
    
    
    
    
udf_take_range = udf(take_range,StringType())

df = df.withColumn('df_range',udf_take_range(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [54]:
# choose only less
def take_less(text_input) :
    if text_input is None:
        return None
    
    try :
        if re.search(r'^l' , text_input) :
            return text_input
    except AttributeError :
        return None
    
    
udf_take_less = udf(take_less,StringType())

df = df.withColumn('df_less',udf_take_less(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [55]:
# choose only more
def take_more(text_input) :
    if text_input is None:
        return None
    
    try :
        if re.search(r'^m' , text_input) :
            return text_input
    except AttributeError :
        return None
    
    
udf_take_more = udf(take_more,StringType())

df = df.withColumn('df_more',udf_take_more(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [56]:
# choose only spot
def take_spot(text_input) :
    if text_input is None:
        return None
    
    try :
        if re.search(r'([0-9]+\s?-+\s?[0-9]+)', text_input) :
            split = text_input.split(' - ')
            
            min_value = str(split[0])
            max_value = str(split[1])

            if min_value == max_value :
                return max_value
            
        elif re.search(r'(^[l,m])' , text_input) :
            return None
        elif re.search(r'([0-9a-zA-Z]{1,10}\s?-\s?[0-9a-zA-Z]{1,10})' , text_input) :
            return None
        elif re.search(r'^[0-9]+[a-zA-Z]?$' , text_input) :
            return text_input
    
        else :
            return None
    except AttributeError :
        return None
    
    
    
    
    
udf_take_spot = udf(take_spot,StringType())

df = df.withColumn('df_spot',udf_take_spot(col(data_transform)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [57]:
# create data frame for range , less , more , spot
income_range = df.filter(df['df_range'].isNotNull())
income_less = df.filter(df['df_less'].isNotNull())
income_more = df.filter(df['df_more'].isNotNull())
income_spot = df.filter(df['df_spot'].isNotNull())

all_incomes = income_range.unionAll(income_less).unionAll(income_more).unionAll(income_spot)
income_unknown = df.exceptAll(all_incomes)

df = df.drop('df_range', 'df_less', 'df_more', 'df_spot' , 'income_transform')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [58]:
data_range = 'df_range'
data_spot = 'df_spot'

# standard format of range and spot
def char_range_spot(text_input):
    try:
        if re.search(r'[a-zA-Zก-๛]+', text_input):
            return None
        elif re.search(r'[0-9]{,10}\s?-\s?[0-9]{,10}', text_input):
            return text_input
        elif text_input.startswith(str('0')):
            return str('0')
        elif re.search(r'([0-9]+)', text_input):
            return text_input
        else:
            return None
    except:
        return None

    
    
udf_char_range_spot = udf(char_range_spot, StringType())

# range
income_range = income_range.withColumn('df_range', udf_char_range_spot(col(data_range)))

# spot
income_spot = income_spot.withColumn('df_spot', udf_char_range_spot(col(data_spot)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [59]:
data_less = 'df_less'
data_more = 'df_more'

# standard format of less and more        
def char_less_more(text_input) :
    try :
        result = re.findall(r'([0-9]+)' , text_input)
        return ''.join(result)
    except :
        return None
    
    
udf_char_less_more = udf(char_less_more,StringType())

# less
income_less = income_less.withColumn('df_less',udf_char_less_more(col(data_less)))

# more
income_more = income_more.withColumn('df_more',udf_char_less_more(col(data_more)))    

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [60]:
# validate length 4 <= data <= 10
# all cases except range
def length(text_input) :
    try :
        text_input = re.findall(r'([0-9]+)' , text_input)
        result = ''.join(text_input)
        
        if 4 <= len(str(result)) <= 10 :
            return True
        else :
            return False
    except :
        return False

    
udf_length = udf(length,StringType())

# less
income_less = income_less.withColumn('validate_length',udf_length(col(data_less)))

# more
income_more = income_more.withColumn('validate_length',udf_length(col(data_more)))    

# spot
income_spot = income_spot.withColumn('validate_length',udf_length(col(data_spot)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [61]:
# validate consecutively number
# range
def length_range(text_input) :
    try :
        if re.search(r'([0-9]+\s?-+\s?[0-9]+)', text_input) :
            split = text_input.split(' - ')
            
            min_value = int(split[0])
            max_value = int(split[1])
            
            if 4 <= len(str(min_value)) <= 10 and 4 <= len(str(max_value)) <= 10 :
                return True
            else :
                return False
        else :
            return False
    except :
        return False
    
    
    
udf_length_range = udf(length_range,StringType())

# range
income_range = income_range.withColumn('validate_length',udf_length_range(col(data_range)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [62]:
# segment range of data
# min_value -> min , max_value -> max
def convert_segment_range(text_input):
    try:
        split = text_input.split(' - ')
        min_value = int(split[0])
        max_value = int(split[1])

        if min_value < max_value:
            return str(min_value), str(max_value), None
        elif min_value == max_value:
            return None, None, str(max_value)
        else:
            return None, None, None
    except:
        return None, None, None
    

schema = StructType([
    StructField("min", StringType(), True),
    StructField("max", StringType(), True),
    StructField("spot", StringType(), True)
])
    
udf_convert_segment_range = udf(convert_segment_range, schema)

income_range = income_range.withColumn("segment_data", udf_convert_segment_range(col(data_range)))
income_range = income_range.withColumn("range_min", income_range["segment_data.min"])
income_range = income_range.withColumn("range_max", income_range["segment_data.max"])
income_range = income_range.withColumn("spot", income_range["segment_data.spot"])

# Drop the intermediate struct column if needed
income_range = income_range.drop("segment_data")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [63]:
# segment less , more , spot of data
# less -> less
# more -> more
# spot -> spot
def convert_segment(text_input) :
    if re.search(r'\s' , text_input) :
        return None
    elif re.search(r'^[0]{1}$' , text_input) :
        return text_input
    elif re.findall(r'[0-9]+' , text_input) :
        return text_input

    
    
    
udf_convert_segment = udf(convert_segment,StringType())

# less
income_less = income_less.withColumn('less',udf_convert_segment(col(data_less)))

# more
income_more = income_more.withColumn('more',udf_convert_segment(col(data_more)))    

# spot
income_spot = income_spot.withColumn('spot',udf_convert_segment(col(data_spot)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [64]:
data_seg_range_min = 'range_min'
data_seg_range_max = 'range_max'
data_seg_less = 'less'
data_seg_more = 'more'
data_seg_spot = 'spot'

# income end with 99 or 1
def income_99(text_input) :
    if text_input == None :
        return None
    elif text_input.endswith('99') :
        result = int(text_input) + 1
        return int(result)
    elif text_input.endswith('1') :
        result = int(text_input) - 1
        return int(result)

    else :
        return text_input
    

    
udf_income_99 = udf(income_99,StringType())

# range_min
income_range = income_range.withColumn('range_min',udf_income_99(col(data_seg_range_min)))

# range_max
income_range = income_range.withColumn('range_max',udf_income_99(col(data_seg_range_max)))

# less
income_less = income_less.withColumn('less',udf_income_99(col(data_seg_less)))

# more
income_more = income_more.withColumn('more',udf_income_99(col(data_seg_more)))    

# spot
income_spot = income_spot.withColumn('spot',udf_income_99(col(data_seg_spot)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [65]:
income_range = income_range.withColumn("less", lit(None)).withColumn("more", lit(None))
income_less = income_less.withColumn("more", lit(None)).withColumn("spot", lit(None)).withColumn("range_min", lit(None)).withColumn("range_max", lit(None))
income_more = income_more.withColumn("less", lit(None)).withColumn("range_min", lit(None)).withColumn("range_max", lit(None)).withColumn("spot", lit(None))
income_spot = income_spot.withColumn("range_min", lit(None)).withColumn("range_max", lit(None)).withColumn("less", lit(None)).withColumn("more", lit(None))
income_unknown = income_unknown.withColumn("range_min", lit(None)).withColumn("range_max", lit(None)).withColumn("less", lit(None)).withColumn("more", lit(None)).withColumn("spot", lit(None)).withColumn("validate_length", lit(None))


selected_cols = ['customer_id', 'income', 'range_min', 'range_max', 'less', 'more', 'spot', 'validate_length']

income = (income_range.select(*selected_cols)
          .unionAll(income_less.select(*selected_cols))
          .unionAll(income_more.select(*selected_cols))
          .unionAll(income_spot.select(*selected_cols))
          .unionAll(income_unknown.select(*selected_cols)).dropDuplicates(["customer_id"]))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [66]:
data_range_min = 'range_min'
data_range_max = 'range_max'

# validate consecutive number of range
def dup_number_range(min_value , max_value):
    try :
        digits_min = set(str(min_value))
        digits_max = set(str(max_value))
        
        if min_value == None and max_value == None :
            return False
        elif len(digits_min) == 1 and len(digits_max) == 1 :
            return False
        elif len(digits_min) >= 2 and len(digits_max) >= 2 :
            return False
        else:
            return True
    except :
        return False
    
    

udf_dup_number_range = udf(dup_number_range,StringType())

# range
income = income.withColumn('validate_consecutive',udf_dup_number_range(col(data_range_min) , col(data_range_max)))


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [67]:
data_less = 'less'
data_more = 'more'
data_spot = 'spot'

# validate consecutive number of data
def dup_number(text_input):
    try :
        digits = set(str(text_input))
        
        if text_input == None :
            return False
        elif len(text_input) == 1 :
            return False
        elif len(digits) >= 2:
            return False
        else:
            return True
    except :
        return False
    
    

udf_dup_number = udf(dup_number,StringType())

# less
income = income.withColumn('validate_consecutive',udf_dup_number(col(data_less)))

# more
income = income.withColumn('validate_consecutive',udf_dup_number(col(data_more)))
                           
# more
income = income.withColumn('validate_consecutive',udf_dup_number(col(data_spot)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [68]:
data_range_min = 'range_min'
data_range_max = 'range_max'
data_less = 'less'
data_more = 'more'
data_spot = 'spot'

# standard format
def combined_income(range_min, range_max, less, more, spot):
    if range_min is not None and range_max is not None:
        result_range = f'{range_min} - {range_max}'
        return result_range
    
    if less is not None and not pd.isna(less):
        result_less = f'less than {less}'
        return result_less

    if more is not None and not pd.isna(more):
        result_more = f'more than {more}'
        return result_more

    if spot is not None:
        result_spot = f'{spot}'
        return result_spot
    
    

udf_combined_income = udf(combined_income,StringType())

income = income.withColumn('standard_income',udf_combined_income(col(data_range_min) , col(data_range_max) , col(data_less) , col(data_more) , col(data_spot)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [69]:
data = 'income'
data_standard = 'standard_income'

# standard output / status
def result(text_input, standard):
    if text_input is None and standard is None:
        return text_input, 3
    elif isinstance(text_input, str) and isinstance(standard, str):
        if text_input == standard:
            return text_input, 1
        else:
            return standard, 2
    elif isinstance(text_input, str) and standard == None :
        return text_input, 0
    

    
schema = StructType([
    StructField("std_output", StringType(), True),
    StructField("status", IntegerType(), True)
])

udf_result = udf(result, schema)

# Apply the UDF to your DataFrame
income = income.withColumn("segment_data", udf_result(col(data), col(data_standard)))

# Split the struct column into separate columns
income = income.withColumn("std_output", income["segment_data.std_output"])
income = income.withColumn("status", income["segment_data.status"])

# Drop the intermediate struct column
income = income.drop("segment_data")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [70]:
selected_cols = ['customer_id' , 'income' , 'range_min' , 'range_max' , 'less' , 'more' , 'spot' , 'standard_income' , 'std_output' , 'status' , 'validate_length' ,'validate_consecutive']

income = income.select(*selected_cols)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
income.show()

## birth date

In [72]:
data = 'birth_date'

# lower case 
# remove special char / days
# convert thai num to arabic num
def standard_transform(text_input):
    text_input = str(text_input)
    
    if text_input == None :
            return '-'
        
    text_input = str(text_input).lower().replace('_' , ' ').replace('.' , '').replace('-' , ' ')
    # [^ก-๛a-z\w\s:]
    text_input = re.sub(r'[^\u0E00-\u0E7F\w\s:]', ' ', text_input)
    
    days = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday","mon","tue","wed","thu","fri","sat","sun",
            # "วันจันทร์","วันอังคาร","วันพุธ","วันพฤหัสบดี","วันศุกร์","วันเสาร์","วันอาทิตย์"
            "\u0e27\u0e31\u0e19\u0e08\u0e31\u0e19\u0e17\u0e23\u0e4c",
            "\u0e27\u0e31\u0e19\u0e2d\u0e31\u0e07\u0e04\u0e32\u0e23",
            "\u0e27\u0e31\u0e19\u0e1e\u0e38\u0e18",
            "\u0e27\u0e31\u0e19\u0e1e\u0e24\u0e2b\u0e31\u0e2a\u0e1a\u0e14\u0e35",
            "\u0e27\u0e31\u0e19\u0e28\u0e38\u0e01\u0e23\u0e4c",
            "\u0e27\u0e31\u0e19\u0e40\u0e2a\u0e32\u0e23\u0e4c",
            "\u0e27\u0e31\u0e19\u0e2d\u0e32\u0e17\u0e34\u0e15\u0e22\u0e4c",
            # "จ", "อ", "พ", "พฤ", "ศ", "ส", "อา"
            "\u0e08", "\u0e2d", "\u0e1e", "\u0e1e\u0e24", "\u0e28", "\u0e2a", "\u0e2d\u0e32"]

    for day in days :
        if day in text_input :
            text_input = text_input.replace(day , '')
            return text_input
    
    # ๐๑๒๓๔๕๖๗๘๙
    thai_digits = '\u0E50\u0E51\u0E52\u0E53\u0E54\u0E55\u0E56\u0E57\u0E58\u0E59'
    western_digits = '0123456789'
    text_input = text_input.translate(str.maketrans(thai_digits, western_digits))
    
    return text_input




udf_standard_transform = udf(standard_transform,StringType())

df = df.withColumn('date_of_birth_transform',udf_standard_transform(col(data)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [73]:
data_transform = 'date_of_birth_transform'

# create time column
def take_time(text_input) :
    try :
        result = text_input.lower()
        result = re.search(r'([0-9]{,2}:[0-9]{,2}:[0-9]{,2}|[0-9]{,2}:[0-9]{,2}\s?[pm|am|\u0E19]{,2})' , text_input).group()
        return result
    except AttributeError :
        return '-'
    
    
    
    
udf_take_time = udf(take_time,StringType())

# create column for time
df = df.withColumn('time',udf_take_time(col(data_transform)))

# create column for date : if the data be in time , not be in date
df = df.withColumn('date', expr("regexp_replace(date_of_birth_transform, time, '')"))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [74]:
data_date = 'date'

# convert char month to num month
def convert_month(text_input):
    text_input = str(text_input)

    if text_input is None:
        return None

    for pattern, value in json_content_birth_date.items():
        if re.search(pattern, text_input):
            result = re.sub(pattern, ' ' + value + ' ', text_input)
            return result
    return text_input

    
    
udf_convert_month = udf(convert_month,StringType())

# create column for convert char month(Jan,...,Dec) to num month(01 - 12)
df = df.withColumn('date', udf_convert_month(col(data_date)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [75]:
# remove text in date            
def remove_text(text_input) :
    # [ก-๛]|[a-z]
    if re.search(r'[a-zA-Z\u0E00-\u0E7F]' , text_input) :
        return None
    else :
        return text_input
    
    
    
    
udf_remove_text = udf(remove_text,StringType())

df = df.withColumn('date',udf_remove_text(col(data_date)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [76]:
# tranform other format to d-m-y
def transform_date_format(text_input):
    if text_input is None:
        return None
    
    text_input = str(text_input)
    
    formats = ['%m %d %Y', '%Y %d %m', '%d %m %Y', '%Y %m %d', '%m %d %y', '%y %d %m', '%d %m %y', '%y %m %d']
    
    for date_format in formats :
        try:
            text_input = text_input.strip()
            date_obj = datetime.datetime.strptime(text_input, date_format)
            result = date_obj.strftime('%d-%m-%Y')
            return result
        except ValueError:
            pass
    
    return None
    
    
    
    
udf_transform_date_format = udf(transform_date_format,StringType())

# transform format
df = df.withColumn('standard_date_of_birth',udf_transform_date_format(col(data_date)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [77]:
data_standard = 'standard_date_of_birth'

# conver year era to christian year era
def convert_year_era(text_input):
    try:
        text_input = str(text_input)
        
        split = text_input.split('-')
        
        if len(split) != 3:
            return None
        
        day, month, year_bs = split
        
        current_date = datetime.date.today()
        current_year_ad = current_date.year
        current_year_bs = current_year_ad + 543 - 200
        
        if len(year_bs) == 4 and int(year_bs) >= current_year_bs:
            christian_year = int(year_bs) - 543
            return f"{day}-{month}-{christian_year}"
        else:
            return f"{day}-{month}-{year_bs}"
    except Exception as e:
        return None
    
    
    
    
udf_convert_year_era = udf(convert_year_era,StringType())

# convert buddhist year era to christian year era
df = df.withColumn('standard_date_of_birth',udf_convert_year_era(col(data_standard)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [78]:
# calculate age
def check_age(text_input):
    try:
        split = text_input.split('-')
        
        for year in split:
            if len(year) == 4:
                birth_year = int(year)
                current_year = datetime.datetime.now().year
                
                if birth_year > current_year:
                    return None
                else:
                    age = current_year - birth_year
                    if age >= 0 :
                        return str(int(age))
                    else :
                        return None
            elif text_input == None :
                return None

    except:
        return None
    
    
    
    
udf_check_age = udf(check_age,IntegerType())

# create column for age
df = df.withColumn('age',udf_check_age(col(data_standard)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [79]:
# validate year of birth < current year
def check_year(text_input) :
    try:
        split = text_input.split('-')
        
        for year in split:
            if len(year) == 4:
                birth_year = int(year)
                current_year = datetime.datetime.now().year

                if birth_year < current_year :
                    return True
                else :
                    return False
    except:
        return None
    
    
    
    
    
udf_check_year = udf(check_year,StringType())

# create column for age
df = df.withColumn('validate_year_of_birth',udf_check_year(col(data_standard)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [80]:
data_age = 'age'

# validate age < 100
def age_less_100(text_input) :
    try :
        if int(text_input) < 100 :
            return True
        else :
            return False
    except :
        return None
    
    
    
    
udf_age_less_100 = udf(age_less_100,StringType())

# create column for validate age < 100
df = df.withColumn('validate_age',udf_age_less_100(col(data_age)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [81]:
data_standard = 'standard_date_of_birth'

# validate standard format
def validate_standard_format(text_input) :
        if isinstance(text_input, datetime.date) and re.search(r'^[0-9]{2}\-[0-9]{2}\-[0-9]{4}$' , text_input) :
            return True
        else :
            return False
        
        
        
udf_validate_standard_format = udf(validate_standard_format,StringType())

df = df.withColumn('validate_standard_format',udf_validate_standard_format(col(data_standard)))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [82]:
selected_cols = ['customer_id','birth_date','date_of_birth_transform','standard_date_of_birth','age','validate_year_of_birth','validate_age']

date_of_birth = df[selected_cols]

df = df.drop('date_of_birth_transform','standard_date_of_birth','age','validate_year_of_birth','validate_age', 'time', 'date')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [83]:
data = 'birth_date'
data_standard = 'standard_date_of_birth'

# standard output / status
def result(text_input, standard):
    if text_input is None and standard is None:
        return (text_input, 3)
    
    text_input = str(text_input)
    standard = str(standard)
    
    if isinstance(text_input, str) and isinstance(standard, str):
        if text_input == standard:
            return (text_input, 1)
        else:
            return (standard, 2)
    elif isinstance(text_input, str) and standard is None:
        return (text_input, 0)



schema = StructType([
    StructField("std_output", StringType(), True),
    StructField("status", IntegerType(), True)
])

udf_result = udf(result, schema)

date_of_birth = date_of_birth.withColumn("segment_data", udf_result(col(data), col(data_standard)))

# Split the struct column into separate columns
date_of_birth = date_of_birth.withColumn("std_output", date_of_birth["segment_data.std_output"])
date_of_birth = date_of_birth.withColumn("status", date_of_birth["segment_data.status"])

# Drop the intermediate struct column
date_of_birth = date_of_birth.drop("segment_data")

# convert string type to date type
date_of_birth = date_of_birth.withColumn("std_output", to_date(date_of_birth["std_output"], "dd-MM-yyyy"))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [84]:
date_of_birth.dtypes

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[('customer_id', 'bigint'), ('birth_date', 'date'), ('date_of_birth_transform', 'string'), ('standard_date_of_birth', 'string'), ('age', 'int'), ('validate_year_of_birth', 'string'), ('validate_age', 'string'), ('std_output', 'date'), ('status', 'int')]

In [ ]:
date_of_birth.show()

## combined all type of dataframe

In [86]:
col_gender = gender.select('customer_id', 'gender','standard_gender' , 'std_output' , 'status')
col_marital_status = marital_status.select('customer_id', 'marital_status','standard_marital_status' , 'std_output' , 'status')
col_education = education.select('customer_id', 'education','standard_education' , 'std_output' , 'status')
col_occupation = occupation.select('customer_id', 'occupation','standard_occupation' , 'std_output' , 'status')
col_nationality = nationality.select('customer_id', 'nationality','standard_nationality' , 'std_output' , 'status')
col_income = income.select('customer_id', 'income','standard_income' , 'std_output' , 'status')
col_birth_date = date_of_birth.select('customer_id', 'birth_date','standard_date_of_birth' , 'std_output' , 'status')


customer = (col_gender
                 .join(col_marital_status, "customer_id", "inner")
                 .join(col_education, "customer_id", "inner")
                 .join(col_occupation, "customer_id", "inner")
                 .join(col_nationality, "customer_id", "inner")
                 .join(col_income, "customer_id", "inner")
                 .join(col_birth_date, "customer_id", "inner"))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [169]:
customer.count()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Interrupted by user


In [ ]:
customer.show(truncate=False)

## count status

In [88]:
column_name = ['status']

# gender
result = gender.groupBy(column_name).agg(count("*").alias("count"))
result.show()


# marital_status
result = marital_status.groupBy(column_name).agg(count("*").alias("count"))
result.show()


# education
result = education.groupBy(column_name).agg(count("*").alias("count"))
result.show()



# occupation
result = occupation.groupBy(column_name).agg(count("*").alias("count"))
result.show()



# nationality
result = nationality.groupBy(column_name).agg(count("*").alias("count"))
result.show()


# income
result = income.groupBy(column_name).agg(count("*").alias("count"))
result.show()


# date_of_birth
result = date_of_birth.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+-------+
|status|  count|
+------+-------+
|     1|     66|
|     3|1668070|
|     2|1038627|
|     0| 118003|
+------+-------+

+------+-------+
|status|  count|
+------+-------+
|     1|   4822|
|     3|2495236|
|     0| 324708|
+------+-------+

+------+-------+
|status|  count|
+------+-------+
|     3|2824766|
+------+-------+

+------+-------+
|status|  count|
+------+-------+
|     1| 161850|
|     3| 175486|
|     2|2165350|
|     0| 322080|
+------+-------+

+------+-------+
|status|  count|
+------+-------+
|     3|2811140|
|     2|  13626|
+------+-------+

+------+-------+
|status|  count|
+------+-------+
|     1| 217893|
|     3|2280784|
|     2| 207310|
|     0| 118779|
+------+-------+

+------+-------+
|status|  count|
+------+-------+
|     3|2543240|
|     2| 281526|
+------+-------+

## data validate

#### count null - not null

In [89]:
columns_to_count = ["birth_date", "gender", "marital_status", "income", "occupation", "nationality", "education"]

# Create a list to store the count expressions
count_expressions = []

# Loop through the specified columns and count null values
for column in columns_to_count:
    count_expr = count(when(df[column].isNull(), True)).alias(f"{column}_null")
    count_expressions.append(count_expr)

# Select and show the result DataFrame
dfs_null = df.select(*count_expressions)
dfs_null.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------+-----------+-------------------+-----------+---------------+----------------+--------------+
|birth_date_null|gender_null|marital_status_null|income_null|occupation_null|nationality_null|education_null|
+---------------+-----------+-------------------+-----------+---------------+----------------+--------------+
|        2543240|    1668070|            2495236|    2280784|         175486|         2811140|       2824766|
+---------------+-----------+-------------------+-----------+---------------+----------------+--------------+

In [90]:
columns_to_count = ["birth_date", "gender", "marital_status", "income", "occupation", "nationality", "education"]

# Create a list to store the count expressions
count_expressions = []

# Loop through the specified columns and count non-null values
for column in columns_to_count:
    count_expr = count(when(col(column).isNotNull(), True)).alias(f"{column}_not_null")
    count_expressions.append(count_expr)

# Select and show the result DataFrame
dfs_not_null = df.select(*count_expressions)
dfs_not_null.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------------+---------------+-----------------------+---------------+-------------------+--------------------+------------------+
|birth_date_not_null|gender_not_null|marital_status_not_null|income_not_null|occupation_not_null|nationality_not_null|education_not_null|
+-------------------+---------------+-----------------------+---------------+-------------------+--------------------+------------------+
|             281526|        1156696|                 329530|         543982|            2649280|               13626|                 0|
+-------------------+---------------+-----------------------+---------------+-------------------+--------------------+------------------+

#### count gender

In [91]:
# gender
# before

column_name = ['gender']

# Perform the count and display the results
result = df.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+-------+
| gender|  count|
+-------+-------+
|   null|1668070|
|      M| 761299|
|    หญิ|      5|
|   male|     29|
|    man|      1|
|unknown| 118003|
| female|     37|
|      F| 277315|
|    ชาย|      4|
| maleชา|      1|
|femaleห|      1|
| Female|      1|
+-------+-------+

In [92]:
# gender
# after

column_name = ['standard_gender']

# Perform the count and display the results
result = gender.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------+-------+
|standard_gender|  count|
+---------------+-------+
|           null|1786073|
|           male| 761334|
|         female| 277359|
+---------------+-------+

#### count marital_status

In [93]:
# marital_status
# before

column_name = ['marital_status']

# Perform the count and display the results
result = df.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------+-------+
|marital_status|  count|
+--------------+-------+
|          null|2495236|
|       Married|   1330|
|       unknown| 324708|
|       Widowed|    186|
|        Single|   3306|
+--------------+-------+

In [94]:
# marital_status
# after

column_name = ['standard_marital_status']

# Perform the count and display the results
result = marital_status.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------------------+-------+
|standard_marital_status|  count|
+-----------------------+-------+
|                   null|2819944|
|                Married|   1330|
|                Widowed|    186|
|                 Single|   3306|
+-----------------------+-------+

#### count education

In [95]:
# education
# before

column_name = ['education']

# Perform the count and display the results
result = df.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+-------+
|education|  count|
+---------+-------+
|     null|2824766|
+---------+-------+

In [96]:
# education
# after

column_name = ['standard_education']

# Perform the count and display the results
result = education.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------+-------+
|standard_education|  count|
+------------------+-------+
|              null|2824766|
+------------------+-------+

#### count occupation

In [97]:
# occupation
# before

column_name = ['occupation']

# Perform the count and display the results
result = df.groupBy(column_name).agg(count("*").alias("count"))
result.show(df.count(), False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------------------------------------------+-------+
|occupation                                     |count  |
+-----------------------------------------------+-------+
|null                                           |175486 |
|freelancer                                     |1643   |
|Employee, experience less than 1 year          |3878   |
|officer                                        |83852  |
|Officer with over 4 months of employment       |787    |
|Employee, experience more than 1 year          |44490  |
|Government-owned company                       |2295   |
|unknown                                        |318623 |
|student                                        |52     |
|Business owner, registered more than 3 years   |7517   |
|unemployed                                     |244183 |
|retired                                        |3      |
|other                                          |1896436|
|Public servent                                 |17385  |
|Business owne

In [98]:
# occupation
# after

column_name = ['standard_occupation']

# Perform the count and display the results
result = occupation.groupBy(column_name).agg(count("*").alias("count"))
result.show(df.count(), False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------------------------------------------+-------+
|standard_occupation                            |count  |
+-----------------------------------------------+-------+
|null                                           |497566 |
|Other                                          |1896436|
|Employee, experience less than 1 year          |3878   |
|officer                                        |83852  |
|Officer with over 4 months of employment       |787    |
|Employee, experience more than 1 year          |44490  |
|Government-owned company                       |2295   |
|Unemployed                                     |244183 |
|Freelancer                                     |1643   |
|Business owner, registered more than 3 years   |7517   |
|Student                                        |52     |
|Public servent                                 |17385  |
|Business owner, registered less than 3 years   |618    |
|Business owner with over 1 year of registration|1028   |
|Officer with 

#### count nationality

In [99]:
# nationality
# before

column_name = ['nationality']

# Perform the count and display the results
result = df.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+-------+
|nationality|  count|
+-----------+-------+
|       null|2811140|
|         TH|  13626|
+-----------+-------+

In [100]:
# nationality
# after

column_name = ['standard_nationality']

# Perform the count and display the results
result = nationality.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-------+
|standard_nationality|  count|
+--------------------+-------+
|                null|2811140|
|                Thai|  13626|
+--------------------+-------+

#### count income

In [101]:
# income
# before

column_name = ['income']

# Perform the count and display the results
result = df.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------+-------+
|          income|  count|
+----------------+-------+
|            null|2280784|
|       60K - 80K|   4697|
|         more80K|   8928|
|More than 80,000|   1662|
|       20K - 40K| 127816|
| 15,000 – 19,999|   8238|
| 60,000 – 79,999|    771|
|               0| 217893|
|         unknown| 100730|
|       40K - 60K|  31397|
| 40,000 – 59,999|   2477|
| 20,000 – 39,999|   6562|
|       15K - 20K|  19644|
|         less15K|  13166|
|      0 – 14,999|      1|
+----------------+-------+

In [102]:
# income
# after

column_name = ['standard_income']

# Perform the count and display the results
result = income.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------+-------+
|standard_income|  count|
+---------------+-------+
|more than 80000|  10590|
|           null|2399563|
|  15000 - 20000|  19644|
|              0| 217893|
|  20000 - 40000| 127816|
|  40000 - 60000|  31397|
|  60000 - 80000|   4697|
|less than 15000|  13166|
+---------------+-------+

#### count date of birth

In [103]:
# birth date
# before

column_name = ['birth_date']

# Perform the count and display the results
result = df.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+-----+
|birth_date|count|
+----------+-----+
|1976-06-24|   10|
|1981-03-25|   19|
|1969-01-04|    7|
|1970-05-29|    4|
|1967-05-31|    6|
|1971-05-12|    6|
|1968-06-15|    9|
|1975-10-24|   10|
|1980-09-23|   17|
|1961-11-01|    4|
|1993-08-26|   13|
|1999-02-01|    5|
|1959-09-30|    4|
|1991-07-30|   15|
|1987-05-29|   15|
|1973-09-04|   10|
|1958-08-15|    4|
|1995-12-01|    8|
|1982-05-17|   13|
|1962-04-18|    3|
+----------+-----+
only showing top 20 rows

In [104]:
# birth date
# after

# validate standard format
column_name = ['standard_date_of_birth']

# Perform the count and display the results
result = date_of_birth.groupBy(column_name).agg(count("*").alias("count"))
result.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------------+-----+
|standard_date_of_birth|count|
+----------------------+-----+
|            06-09-1992|   16|
|            06-03-1991|   18|
|            14-05-1966|    4|
|            15-07-1974|    8|
|            31-08-1993|   20|
|            22-10-2002|   79|
|            21-04-1991|   16|
|            05-12-1977|    8|
|            12-03-1992|    8|
|            20-06-1982|    3|
|            09-05-1994|   11|
|            07-03-1991|   12|
|            15-05-1983|    7|
|            11-10-1976|    8|
|            13-01-1986|   12|
|            01-12-1989|   20|
|            07-05-1990|   10|
|            04-08-1960|    4|
|            03-10-1988|   14|
|            13-01-1978|   14|
+----------------------+-----+
only showing top 20 rows

## count distrinct

In [105]:
# distrinct

#########
# gender
unique_row_count_b = df.select("gender").distinct().count()
unique_row_count_a = gender.select("standard_gender").distinct().count()

print('distinct count of gender before :', unique_row_count_b)
print('distinct count of gender after :', unique_row_count_a)

########
# marital status
unique_row_count_b = df.select("marital_status").distinct().count()
unique_row_count_a = marital_status.select("standard_marital_status").distinct().count()

print('distinct count of marital_status before :', unique_row_count_b)
print('distinct count of marital_status after :', unique_row_count_a)

########
# education
unique_row_count_b = df.select("education").distinct().count()
unique_row_count_a = education.select("standard_education").distinct().count()

print('distinct count of education before :', unique_row_count_b)
print('distinct count of education after :', unique_row_count_a)

########
# occupation
unique_row_count_b = df.select("occupation").distinct().count()
unique_row_count_a = occupation.select("standard_occupation").distinct().count()

print('distinct count of occupation before :', unique_row_count_b)
print('distinct count of occupation after :', unique_row_count_a)

########
# nationality
unique_row_count_b = df.select("nationality").distinct().count()
unique_row_count_a = nationality.select("standard_nationality").distinct().count()

print('distinct count of nationality before :', unique_row_count_b)
print('distinct count of nationality after :', unique_row_count_a)

########
# income
unique_row_count_b = df.select("income").distinct().count()
unique_row_count_a_rmin = income.select("range_min").distinct().count()
unique_row_count_a_rmax = income.select("range_max").distinct().count()
unique_row_count_a_l = income.select("less").distinct().count()
unique_row_count_a_m = income.select("more").distinct().count()
unique_row_count_a_s = income.select("spot").distinct().count()

print('distinct count of income before :', unique_row_count_b)
print('distinct count of income range min after :', unique_row_count_a_rmin)
print('distinct count of income max before :', unique_row_count_a_rmax)
print('distinct count of income less after :', unique_row_count_a_l)
print('distinct count of income more before :', unique_row_count_a_m)
print('distinct count of income spot after :', unique_row_count_a_s)

########
# date of birth
unique_row_count_b = df.select("birth_date").distinct().count()
unique_row_count_a = date_of_birth.select("standard_date_of_birth").distinct().count()

print('distinct count of birth_date before :', unique_row_count_b)
print('distinct count of birth_date after :', unique_row_count_a)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

distinct count of gender before : 12
distinct count of gender after : 3
distinct count of marital_status before : 5
distinct count of marital_status after : 4
distinct count of education before : 1
distinct count of education after : 1
distinct count of occupation before : 19
distinct count of occupation after : 17
distinct count of nationality before : 2
distinct count of nationality after : 2
distinct count of income before : 15
distinct count of income range min after : 5
distinct count of income max before : 5
distinct count of income less after : 2
distinct count of income more before : 2
distinct count of income spot after : 2
distinct count of birth_date before : 17459
distinct count of birth_date after : 17396